# Quickstart: Trace an agent
# 快速开始：追踪一个 agent

Trace a multi-turn agent with the Weave SDK. Sessions, turns, LLM calls, and tool calls render in the Agents view of your project.

使用 Weave SDK 追踪一个多轮 agent。session、turn、LLM 调用和工具调用都会显示在项目的 Agents 视图里。

The Weave SDK allows you to trace custom agents or agents created using popular SDKs. This quickstart guides you through how to manually integrate Weave into a custom-built multi-turn agent to emit and capture OpenTelemetry spans and render them in Weave's Agents view.

Weave SDK 可以追踪你自己写的 agent，也可以追踪用常见 SDK 创建的 agent。本教程演示如何把 Weave 手动接入一个自定义多轮 agent，发出并捕获 OpenTelemetry spans，最后在 Weave 的 Agents 视图中查看。

If you are looking to integrate Weave with popular SDKs or harnesses, such as the Claude Agents SDK or Codex, see the [Weave integration section](https://docs.wandb.ai/weave/guides/integrations). Weave autopatches into several popular agent-building SDKs and agent harnesses for quick integration.

如果你想把 Weave 接到 Claude Agents SDK、Codex 这类现成 SDK 或 harness，可以看 [Weave integration section](https://docs.wandb.ai/weave/guides/integrations)。Weave 对一些常见 agent 框架支持 autopatch，可以更快接入。

## What you'll learn
## 你会学到什么

The code in this guide sets up a small research agent that can look things up on Wikipedia. It asks three questions (three turns), lets the AI decide when to search Wikipedia for an answer, and uses Weave to record every step (the conversation, each question, each AI response, and each Wikipedia lookup) so you can see exactly what happened in the Weave Agents view.

这个示例会搭一个很小的 research agent，它可以查 Wikipedia。它会连续问三个问题，也就是三个 turn，让 AI 自己决定什么时候调用 Wikipedia 工具，并用 Weave 记录每一步：对话、用户问题、AI 回复、Wikipedia 查询。这样你可以在 Weave Agents 视图里看到完整过程。

This guide shows you how to:

这个教程会展示：

- Initialize Weave for agent tracing with `weave.init()`
- 用 `weave.init()` 初始化 Weave agent tracing
- Open a session and a turn with `weave.start_session()` and `weave.start_turn()`
- 用 `weave.start_session()` 和 `weave.start_turn()` 开启 session 和 turn
- Wrap LLM calls with `weave.start_llm()` and record usage
- 用 `weave.start_llm()` 包住 LLM 调用，并记录 token usage
- Wrap tool executions with `weave.start_tool()` and record results
- 用 `weave.start_tool()` 包住工具调用，并记录工具结果
- View the resulting session, turns, and tool calls in the Agents view
- 在 Agents 视图里查看 session、turn 和 tool call

## How the Weave SDK works with agents
## Weave SDK 如何和 agents 配合

The Weave SDK includes a generic OTel ingest system for agents, meaning that Weave can capture information from any OTel span in your agent's code. However, Weave requires special handling of the following spans to render your agent's traces in the Agents view of the Weave UI.

Weave SDK 内置了面向 agent 的通用 OTel ingest 系统，也就是说 Weave 可以捕获你 agent 代码里的 OTel span。不过为了在 Weave UI 的 Agents 视图里正确渲染，下面这些 span 需要特殊处理。

| Function | Maps to | OTel span |
| --- | --- | --- |
| `weave.start_session(...)` | A conversation | (no span — groups turns) |
| `weave.start_turn(...)` | One user / agent exchange | `invoke_agent` |
| `weave.start_llm(...)` | One LLM API call | `chat` |
| `weave.start_tool(...)` | One tool execution | `execute_tool` |

All four are context managers. On exit, they end the span and flush attributes, including on exceptions.

这四个都是 context manager。退出 `with` 块时，它们会结束 span 并 flush attributes；即使发生异常也会记录。

Other [GenAI semantic-convention attributes](https://opentelemetry.io/docs/specs/semconv/gen-ai/gen-ai-agent-spans/), such as `gen_ai.usage.*` and `gen_ai.agent.name`, enable additional rendering, but they are optional.

其他 [GenAI semantic-convention attributes](https://opentelemetry.io/docs/specs/semconv/gen-ai/gen-ai-agent-spans/)，例如 `gen_ai.usage.*` 和 `gen_ai.agent.name`，可以启用更多 UI 展示能力，但不是必需的。

## Prerequisites
## 前置条件

- A W&B account and [API key](https://wandb.ai/authorize)
- 一个 W&B 账号和 [API key](https://wandb.ai/authorize)
- Python 3.10+
- Python 3.10+
- An OpenAI API key
- 一个 OpenAI API key

## Install packages
## 安装依赖

Install the following packages into your developer environment:

把下面这些依赖安装到你的开发环境里：

In [1]:
print("Dependencies ready")

Dependencies ready


## Initialize Weave
## 初始化 Weave

`weave.init()` authenticates with W&B and configures the OTel exporter that sends agent spans to the **Agents** view. If the project does not exist on your team, Weave creates it the first time you write to it.

`weave.init()` 会完成 W&B 认证，并配置 OTel exporter，把 agent spans 发送到 **Agents** 视图。如果团队下还没有这个项目，Weave 会在第一次写入时自动创建。这个版本会优先读取仓库根目录的 `.env`，避免每次 notebook 都手动输入 key。

In [2]:
import os
from dotenv import load_dotenv
import weave

load_dotenv(override=True)
weave.init(f"{os.environ['WANDB_ENTITY']}/{os.environ['WANDB_PROJECT']}")


weave: Logged in as Weights & Biases user: tian-lu.
weave: View Weave data at https://wandb.ai/tian-lu-university-of-california/weavehacks4-your-idea/weave


## Define a tool
## 定义一个工具

The following code defines the agent's Wikipedia search tool and an OpenAI tool schema to determine when and how to use the tool.

下面的代码定义了 agent 的 Wikipedia 搜索工具，以及 OpenAI tool schema。schema 会告诉模型：这个工具叫什么、做什么、需要什么参数。

In [3]:
import json
import requests

def wikipedia_search(query: str) -> str:
    r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query", "generator": "search", "gsrsearch": query, "gsrlimit": 1,
            "prop": "extracts", "exintro": True, "explaintext": True, "format": "json",
        },
        headers={"User-Agent": "weave-demo"},
    ).json()
    return next(iter(r["query"]["pages"].values()))["extract"]

wikipedia_tool_schema = {
    "type": "function",
    "function": {
        "name": "wikipedia_search",
        "description": "Search Wikipedia for a topic and return its intro paragraph.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}

## Run a traced multi-turn agent
## 运行一个被追踪的多轮 agent

The example below runs three turns in a single session. Each turn:

下面的例子会在一个 session 里运行三个 turn。每个 turn 会：

1. Opens a `chat` span and, for the first two factual questions, forces the LLM to call the Wikipedia tool
1. 打开一个 `chat` span，并对前两个事实性问题强制 LLM 调用 Wikipedia 工具
2. If the LLM requested a tool, opens an `execute_tool` span around the call and feeds the result back to the LLM
2. 如果 LLM 请求调用工具，就打开一个 `execute_tool` span，执行工具，并把结果喂回给 LLM
3. Opens a second `chat` span to produce the final answer
3. 再打开第二个 `chat` span，让 LLM 基于工具结果生成最终答案

In [4]:
from openai import OpenAI

openai_client = OpenAI()
MODEL = os.environ["OPENAI_MODEL"]

def run_turn(history, user_message, use_tool=False):
    history.append({"role": "user", "content": user_message})

    with weave.start_turn(user_message=user_message, model=MODEL):
        # LLM call 1 — for factual questions, force a Wikipedia tool call.
        # 第一次 LLM 调用：对事实性问题，强制调用 Wikipedia 工具。
        with weave.start_llm(model=MODEL, provider_name="openai") as llm:
            request = {"model": MODEL, "messages": history}
            if use_tool:
                request["tools"] = [wikipedia_tool_schema]
                request["tool_choice"] = {"type": "function", "function": {"name": "wikipedia_search"}}

            resp = openai_client.chat.completions.create(**request)
            msg = resp.choices[0].message
            llm.output(msg.content or "")
            llm.usage = weave.Usage(
                input_tokens=resp.usage.prompt_tokens,
                output_tokens=resp.usage.completion_tokens,
            )
            history.append(msg.model_dump(exclude_none=True))

        # If no tool was requested, the first LLM response is the answer.
        # 如果模型没有请求工具调用，第一次 LLM 回复就是最终答案。
        if not msg.tool_calls:
            return msg.content

        # Execute each requested tool call.
        # 执行模型请求的每个工具调用。
        for tc in msg.tool_calls:
            print(f"TOOL: {tc.function.name}({tc.function.arguments})")
            with weave.start_tool(
                name=tc.function.name,
                arguments=tc.function.arguments,
                tool_call_id=tc.id,
            ) as tool:
                tool.result = wikipedia_search(**json.loads(tc.function.arguments))
                history.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": tool.result,
                })

        # LLM call 2 — synthesize the final answer.
        # 第二次 LLM 调用：基于工具结果综合生成最终答案。
        with weave.start_llm(model=MODEL, provider_name="openai") as llm:
            resp = openai_client.chat.completions.create(model=MODEL, messages=history)
            msg = resp.choices[0].message
            llm.output(msg.content)
            llm.usage = weave.Usage(
                input_tokens=resp.usage.prompt_tokens,
                output_tokens=resp.usage.completion_tokens,
            )
            history.append({"role": "assistant", "content": msg.content})
            return msg.content

with weave.start_session(agent_name="research-bot") as session:
    history = []
    questions = [
        ("Who founded Anthropic?", True),
        ("What is Claude (the AI assistant)?", True),
        ("Summarize what we discussed in one sentence.", False),
    ]
    for question, use_tool in questions:
        print(f"USER: {question}")
        print(f"AGENT: {run_turn(history, question, use_tool=use_tool)}\n")


USER: Who founded Anthropic?


weave: 🍩 https://wandb.ai/tian-lu-university-of-california/weavehacks4-your-idea/r/call/019e9ed4-a9a2-7c80-a12a-544dfe0bb42c


TOOL: wikipedia_search({"query":"Anthropic"})


weave: 🍩 https://wandb.ai/tian-lu-university-of-california/weavehacks4-your-idea/r/call/019e9ed4-afd3-7e2c-a11a-3eed6b3a1695


AGENT: Anthropic was founded in 2021 by former OpenAI employees, including siblings **Dario Amodei** and **Daniela Amodei**.

USER: What is Claude (the AI assistant)?


...

TOOL: wikipedia_search({"query":"Claude AI assistant"})


weave: 🍩 https://wandb.ai/tian-lu-university-of-california/weavehacks4-your-idea/r/call/019e9ed4-bb45-7fce-8c0d-d4076e0b73ae


AGENT: **Claude** is an AI assistant and family of large language models developed by **Anthropic**.

It’s designed to help with tasks like:

- Answering questions
- Writing and editing text
- Summarizing documents
- Coding and debugging
- Brainstorming ideas
- Analyzing information
- Conversational assistance

Claude is known for Anthropic’s focus on **AI safety** and “constitutional AI,” an approach intended to make the model more helpful, honest, and harmless.

USER: Summarize what we discussed in one sentence.


AGENT: We discussed that Anthropic was founded in 2021 by former OpenAI employees including Dario and Daniela Amodei, and that Claude is Anthropic’s AI assistant/large language model designed for tasks like answering questions, writing, coding, summarizing, and analysis.



weave: 🍩 https://wandb.ai/tian-lu-university-of-california/weavehacks4-your-idea/r/call/019e9ed4-c951-7ec0-82d3-04290264e19c


## See your agent traces in the Agents view
## 在 Agents 视图查看 agent traces

When `weave.init()` runs, it prints a link to your project where you can see:

`weave.init()` 运行时会打印项目链接。打开后可以看到：

- A row in the **Agents** tab for `research-bot`
- **Agents** tab 里有一行 `research-bot`
- One session containing three turns
- 一个 session，里面包含三个 turns
- Each turn (`invoke_agent`) with two `chat` spans and an `execute_tool` span nested inside
- 每个 turn，也就是 `invoke_agent`，内部嵌套两个 `chat` spans 和一个 `execute_tool` span
- Token counts, latency, model, and the full message exchange on each `chat`
- 每个 `chat` 的 token 数、延迟、模型名和完整 message exchange

Click into any turn to inspect the inputs, outputs, tool arguments, and tool results.

点击任意 turn，就可以检查输入、输出、工具参数和工具结果。

## Next steps
## 下一步

- Get a better understanding of how to [trace agents with Weave](https://docs.wandb.ai/weave/guides/tracking/trace-agents) and what features and options are available in the Weave SDK.
- 继续看 [trace agents with Weave](https://docs.wandb.ai/weave/guides/tracking/trace-agents)，了解 Weave SDK 还有哪些功能和选项。
- See the [integration section](https://docs.wandb.ai/weave/guides/integrations) for more options on how to integrate Weave with your agents.
- 看 [integration section](https://docs.wandb.ai/weave/guides/integrations)，了解如何把 Weave 接到更多 agent 框架。